In [1]:
import pandas as pd
df = pd.read_csv("/home/ibm-user/transactions.csv", nrows=100_000)
print(df.shape)
df.head()

(20000, 13)


,DATE_CREDIT,ENV_NAME,ENV_ID,ENV_TYPE,ENV_CPL_PAN,CARD_BRND,TX_MRCHNT_ID,TX_TOTAL_AMT,CONT_ID,POSTAL_CODE,AGE,AUTHORRESULT,FRAUD_VER_RESULT
0,2013-04-03T00:47:00.001+02:00,BancaItaliana,C0120580917273555,Don't be alone,1009549375,Visa Gold,7273,84.87,1.009549e+09,C012058091,51,APPR,N
1,2013-01-19T18:55:12.001+02:00,BancaItaliana,C0120580915621418,Donna Elegante,1009549375,Visa Gold,5621,179.43,1.009549e+09,C012058091,51,APPR,N
2,2013-06-03T19:58:05.001+02:00,BancaItaliana,C0120580915311606,Rinascimento,1009549375,Visa Gold,5311,140.34,1.009549e+09,C012058091,51,APPR,N
3,2013-06-03T00:02:00.001+02:00,BancaItaliana,C0120580915814746,Eat and Go,1009549375,Visa Gold,5814,144.10,1.009549e+09,C012058091,51,APPR,N
4,2013-03-02T16:00:00.001+02:00,BancaItaliana,C0120580915992763,Florists,1009549375,Visa Gold,5992,111.73,1.009549e+09,C012058091,51,APPR,N


In [2]:
for col in df.columns:
    print(f"{col:20} {df[col].nunique():>7} unique   e.g. {df[col].iloc[0]}")

DATE_CREDIT            14462 unique   e.g. 2013-04-03T00:47:00.001+02:00
ENV_NAME                   1 unique   e.g. BancaItaliana
ENV_ID                  9550 unique   e.g. C0120580917273555
ENV_TYPE                  64 unique   e.g. Don't be alone
ENV_CPL_PAN              113 unique   e.g. 1009549375
CARD_BRND                  3 unique   e.g. Visa Gold
TX_MRCHNT_ID              55 unique   e.g. 7273
TX_TOTAL_AMT            8520 unique   e.g. 84.87
CONT_ID                  112 unique   e.g. 1009549370.0
POSTAL_CODE              101 unique   e.g. C012058091
AGE                       41 unique   e.g. 51
AUTHORRESULT               1 unique   e.g. APPR
FRAUD_VER_RESULT           2 unique   e.g. N


In [3]:
df["Is Fraud?"].value_counts()

KeyError: 'Is Fraud?'

In [4]:
print(df.shape)
df["FRAUD_VER_RESULT"].value_counts()

(20000, 13)


FRAUD_VER_RESULT
N    19998
Y        2
Name: count, dtype: int64

In [5]:
df["ENV_TYPE"].value_counts().head(15)

ENV_TYPE
GasStation              2895
Movies                  2010
Incrocio                1364
AllFood                 1313
TuttoePiu               1258
House of Drapery        1019
Home of Crystal          988
All for your House       987
My Beatiful House        928
Plumbing and Heating     377
Shoes                    363
M and W                  362
FourSeasons              361
Donna Elegante           360
Donna Affascinante       357
Name: count, dtype: int64

In [6]:
import pandas as pd, numpy as np
REPO = "/home/ibm-user/shared/safecircle"

COLS = {"user": "CONT_ID", "amount": "TX_TOTAL_AMT", "merchant": "TX_MRCHNT_ID",
        "time": "DATE_CREDIT", "fraud": "FRAUD_VER_RESULT"}

df = pd.read_csv("/home/ibm-user/transactions.csv", nrows=100_000)
df = df.dropna(subset=[COLS["user"], COLS["amount"]])
df["user_id"] = df[COLS["user"]]
df["amount"] = df[COLS["amount"]].astype(float)
df = df[df["amount"] > 0].copy()

# hour straight from the text "2013-04-03T00:47:..." so time zones don't shift it
df["hour"] = df[COLS["time"]].astype(str).str[11:13].astype(int)

# put each customer's payments in time order before calculating "new payee"
df = df.sort_values(["user_id", COLS["time"]])

usual = df.groupby("user_id")["amount"].transform("median")
df["amount_ratio"] = (df["amount"] / usual).clip(upper=100)
df["new_payee"] = (df.groupby(["user_id", COLS["merchant"]]).cumcount() == 0).astype(int)

FEATURES = ["amount_ratio", "new_payee", "hour"]
X = df[FEATURES]
X.describe()

,amount_ratio,new_payee,hour
count,20000.000000,20000.000000,20000.00000
mean,1.329764,0.148850,13.92675
std,1.440717,0.355949,5.59055
min,0.100882,0.000000,0.00000
25%,0.645351,0.000000,11.00000
50%,1.000000,0.000000,16.00000
75%,1.557651,0.000000,18.00000
max,31.508327,1.000000,21.00000


In [7]:
y = df[COLS["fraud"]].astype(str).str.strip().str.lower().isin(["y", "yes", "1", "true"]).astype(int).values

In [8]:
import os
print(os.path.exists("/home/ibm-user/shared/safecircle/scenarios/history.csv"))
print(os.listdir("/home/ibm-user/shared/safecircle/scenarios"))

False
['transactions.csv', '.ipynb_checkpoints', 'messages.csv']


In [9]:
import pandas as pd, numpy as np
REPO = "/home/ibm-user/shared/safecircle"
rng = np.random.default_rng(42)          # fixed seed so we all get the same file

people = {
    "rosa":   [("Kroger", "grocery", (30, 90)), ("Walgreens", "pharmacy", (10, 40)), ("Corner Cafe", "restaurant", (8, 20))],
    "walter": [("CVS Pharmacy", "pharmacy", (15, 60)), ("Tom Thumb", "grocery", (25, 80)), ("Shell", "gas", (30, 55))],
    "linda":  [("Target", "retail", (20, 120)), ("Kroger", "grocery", (40, 110)), ("Atmos Energy", "utilities", (60, 95))],
}
rows = []
for user, stores in people.items():
    for i in range(12):
        merchant, cat, (lo, hi) = stores[i % len(stores)]
        when = pd.Timestamp("2026-09-01") + pd.Timedelta(days=3 * i, hours=int(rng.integers(9, 18)))
        rows.append({"user_id": user, "timestamp": when.strftime("%Y-%m-%d %H:%M"),
                     "amount": round(float(rng.uniform(lo, hi)), 2),
                     "merchant": merchant, "merchant_category": cat})

hist = pd.DataFrame(rows)
hist.to_csv(f"{REPO}/scenarios/history.csv", index=False)
hist.groupby("user_id")["amount"].median()

user_id
linda     66.230
rosa      34.670
walter    47.125
Name: amount, dtype: float64

In [10]:
import pandas as pd, numpy as np
REPO = "/home/ibm-user/shared/safecircle"

# Fix these names to match what you found in B1
COLS = {"user": "User", "amount": "Amount", "merchant": "Merchant Name",
        "mcc": "MCC", "time": "Time", "fraud": "Is Fraud?"}
RISKY_MCC = {4829, 6051, 6540}   # wire transfer / money orders, quasi-cash, stored-value (gift) cards

df = pd.read_csv("/home/ibm-user/transactions.csv", nrows=100_000)
df["amount"] = (df[COLS["amount"]].astype(str)
                .str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float))
df = df[df["amount"] > 0].copy()                      # drop refunds (negative amounts)
df["user_id"] = df[COLS["user"]]

# Feature 1: amount compared with this person's usual (median) amount
usual = df.groupby("user_id")["amount"].transform("median")
df["amount_ratio"] = (df["amount"] / usual).clip(upper=100)

# Feature 2: new payee = first time this user pays this merchant
# (assumes rows are in time order per user, which TabFormer is)
df["new_payee"] = (df.groupby(["user_id", COLS["merchant"]]).cumcount() == 0).astype(int)

# Feature 3: risky category (gift card, wire, money transfer)
df["risky_category"] = df[COLS["mcc"]].isin(RISKY_MCC).astype(int)

# Feature 4: hour of day, from "HH:MM"
df["hour"] = df[COLS["time"]].astype(str).str.split(":").str[0].astype(int)

FEATURES = ["amount_ratio", "new_payee", "risky_category", "hour"]
X = df[FEATURES]
X.describe()

KeyError: 'Amount'

In [1]:
import shutil, os
WORK = "/home/ibm-user/shared/safecircle"
REPO = "/home/ibm-user/shared/safecircle"

for f in ["src/transactions.py", "src/risk.py",
          "scenarios/history.csv", "models/transaction_model.joblib"]:
    shutil.copy2(f"{WORK}/{f}", f"{REPO}/{f}")
    print("copied", f)

# Scenario CSVs: only copy if Akshi hasn't put the official ones in yet
for f in ["scenarios/messages.csv", "scenarios/transactions.csv"]:
    if not os.path.exists(f"{REPO}/{f}"):
        shutil.copy2(f"{WORK}/{f}", f"{REPO}/{f}")
        print("copied", f, "(tell Akshi)")

shutil.copytree(f"{WORK}/notebooks/pairB", f"{REPO}/notebooks/pairB", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("Untitled*", ".ipynb_checkpoints"))
print("notebooks:", os.listdir(f"{REPO}/notebooks/pairB"))

FileNotFoundError: [Errno 2] No such file or directory: '/home/ibm-user/shared/safecircle/src/transactions.py'

In [2]:
import os, glob
print("In shared:", os.listdir("/home/ibm-user/shared"))
print()
for name in ["transactions.py", "risk.py", "history.csv", "transaction_model.joblib"]:
    hits = glob.glob(f"/home/ibm-user/shared/**/{name}", recursive=True)
    print(name, "->", hits)

In shared: ['Untitled.ipynb', 'safecircle', 'safecircle', '.Trash-1000', 'Untitled2.ipynb', 'Untitled1.ipynb', '.ipynb_checkpoints', 'staging', 'detector.ipynb', 'authors.txt']

transactions.py -> ['/home/ibm-user/shared/safecircle/src/transactions.py']
risk.py -> ['/home/ibm-user/shared/safecircle/src/risk.py']
history.csv -> ['/home/ibm-user/shared/safecircle/scenarios/history.csv', '/home/ibm-user/shared/safecircle/scenarios/history.csv']
transaction_model.joblib -> ['/home/ibm-user/shared/safecircle/models/transaction_model.joblib']
